# CommuniSign — Phase 4b: Phrase LSTM Training

**Input:** `isolated_signs_processed.pkl` (uploaded as Kaggle dataset `qasimmaajid/communisign-processed-isolated`).

**Vocabulary:** 30 curated signs filtered from Google's ASL Signs dataset — hello, thankyou, please, bye, yes, no, mom, dad, brother, grandma, grandpa, drink, sleep, go, look, listen, see, have, finish, wait, happy, sad, hungry, sleepy, food, water, home, dog, cat, now.

**Architecture:** Stacked LSTM (128 → 64) + Dense head. Input is a 30-frame × 63-d landmark sequence. Output is one of 30 signs.

**Outputs (`/kaggle/working/`):**
- `models/phrases_model.h5`, `phrases_model.tflite`, `phrases_label_encoder.pkl`
- `logs/phrases_training.csv`
- `logs/phrases_classification_report.csv`
- `logs/phrases_confusion_matrix.png`, `phrases_training_curves.png`

**Compute:** Settings → Accelerator → **GPU T4 ×2**. ~20–40 min training time.

## 0. Setup
Imports, GPU check, output dirs.

In [ ]:
import os, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

DATA_ROOT = Path('/kaggle/input/datasets/qasimmaajid/communisign-processed-isolated')
OUT  = Path('/kaggle/working')
MODEL_OUT = OUT / 'models'
LOG_OUT   = OUT / 'logs'
MODEL_OUT.mkdir(parents=True, exist_ok=True)
LOG_OUT.mkdir(parents=True, exist_ok=True)

print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Data root exists:', DATA_ROOT.exists())
if DATA_ROOT.exists():
    print('Files:', sorted(p.name for p in DATA_ROOT.iterdir()))

## 1. Load the processed sequences
Each row: `sequence_id`, `participant_id`, `features` (np.ndarray of shape (n_frames, 63)), `n_frames`, `sign` (string label).

In [ ]:
df = pd.read_pickle(DATA_ROOT / 'isolated_signs_processed.pkl')
print('Total sequences:', len(df))
print('Unique signs  :', df['sign'].nunique())
print('\nPer-sign counts:')
print(df['sign'].value_counts().to_string())
print('\nFrame length stats:')
print(df['n_frames'].describe())

## 2. Pad/truncate every sequence to 30 frames
LSTMs need a fixed input length per batch. We pick **30 frames** as the target — close to the 75th percentile of sequence lengths, so most sequences need only minor truncation, while shorter ones are zero-padded.

For sequences longer than 30 frames, we take the **middle 30** (avoids start/end noise where the signer is settling into or out of the sign).

For shorter sequences, we **zero-pad at the end** — the LSTM with masking will treat trailing zeros as 'no input' rather than meaningful frames.

In [ ]:
TARGET_LEN = 30

def pad_or_truncate(frames: np.ndarray, target_len: int = TARGET_LEN) -> np.ndarray:
    n = len(frames)
    if n >= target_len:
        start = (n - target_len) // 2
        return frames[start:start + target_len]
    padded = np.zeros((target_len, frames.shape[1]), dtype=np.float32)
    padded[:n] = frames
    return padded

X_seq = np.stack([pad_or_truncate(s) for s in df['features']])  # (N, 30, 63)
print('Sequence tensor shape:', X_seq.shape, 'dtype:', X_seq.dtype)

## 3. Encode string labels to integers
Same pattern as Phase 4a. Save the encoder so the server can decode predictions back to sign names.

In [ ]:
labels_sorted = sorted(df['sign'].unique())
label_to_idx = {lab: i for i, lab in enumerate(labels_sorted)}
idx_to_label = {i: lab for lab, i in label_to_idx.items()}

y = np.array([label_to_idx[s] for s in df['sign']], dtype=np.int64)
print(f'NUM_CLASSES = {len(labels_sorted)}')
print('Classes:', labels_sorted)

with open(MODEL_OUT / 'phrases_label_encoder.pkl', 'wb') as f:
    pickle.dump({'label_to_idx': label_to_idx, 'idx_to_label': idx_to_label}, f)

## 4. Stratified train/val/test split (80/10/10)
Stratify by sign so every split has every class proportionally.

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(X_seq))
train_idx, temp_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)
val_idx,   test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=y[temp_idx])

X_train, y_train = X_seq[train_idx], y[train_idx]
X_val,   y_val   = X_seq[val_idx],   y[val_idx]
X_test,  y_test  = X_seq[test_idx],  y[test_idx]

print(f'train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}')

## 5. Class weights
Even though the curated 30-sign vocab is balanced (~1.4× ratio), keeping class_weight in the loss makes the model robust to small differences.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
print('Class weights (highest 5):')
for cls, w in sorted(class_weight_dict.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f'  {idx_to_label[cls]:10s} weight={w:.3f}')

## 6. Sequence-aware augmentation
Same idea as Phase 4a (rotation/scale/translation/noise) but applied **once per sequence, identically to all 30 frames** — that's critical: if we rotated each frame independently, we'd destroy the motion pattern that makes a sign recognizable. We rotate the whole sequence as a rigid body, then add per-landmark noise.

In [ ]:
def augment_sequence(features, label):
    # features: (30, 63) — reshape to (30, 21, 3) for spatial ops
    seq = tf.reshape(features, (TARGET_LEN, 21, 3))

    # Rotation in xy plane — same angle for every frame (preserves motion)
    angle = tf.random.uniform([], -0.26, 0.26)  # ~±15°
    cos_a, sin_a = tf.cos(angle), tf.sin(angle)
    x = seq[..., 0] * cos_a - seq[..., 1] * sin_a
    y_ = seq[..., 0] * sin_a + seq[..., 1] * cos_a
    seq = tf.stack([x, y_, seq[..., 2]], axis=-1)

    # Scale — same factor across the sequence
    scale = tf.random.uniform([], 0.9, 1.1)
    seq *= scale

    # Translation — same shift for whole sequence
    shift = tf.random.uniform([3], -0.05, 0.05)
    seq += shift

    # Per-landmark, per-frame noise (small) — simulates MediaPipe jitter
    seq += tf.random.normal(tf.shape(seq), stddev=0.005)

    return tf.reshape(seq, (TARGET_LEN, 63)), label

BATCH = 128
train_ds = (tf.data.Dataset.from_tensor_slices((X_train.astype(np.float32), y_train))
            .map(augment_sequence, num_parallel_calls=tf.data.AUTOTUNE)
            .shuffle(4096).batch(BATCH).prefetch(tf.data.AUTOTUNE))
val_ds   = (tf.data.Dataset.from_tensor_slices((X_val.astype(np.float32), y_val))
            .batch(BATCH).prefetch(tf.data.AUTOTUNE))
print('train_ds element_spec:', train_ds.element_spec)

## 7. Model — stacked LSTM
**Masking** layer first — tells the LSTM to ignore zero-padded timesteps (important for short sequences that got padded).

**LSTM(128, return_sequences=True)** — outputs a 128-d vector for each of the 30 timesteps. Stacked LSTMs benefit from intermediate sequence outputs.

**LSTM(64)** — collapses the sequence to a single 64-d summary vector (it returns only the last timestep's output by default).

**Dense head** — 128 hidden units → softmax over 30 signs.

In [ ]:
NUM_CLASSES = len(labels_sorted)

model = keras.Sequential([
    keras.layers.Input(shape=(TARGET_LEN, 63)),
    keras.layers.Masking(mask_value=0.0),
    keras.layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.0),
    keras.layers.LSTM(64, dropout=0.2, recurrent_dropout=0.0),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3'),
             keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5')],
)
model.summary()

## 8. Train
LSTMs need more epochs and patience than the MLP — set max 100, EarlyStopping patience 12. Same callback pattern otherwise.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=12,
                                   restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                                       min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(str(MODEL_OUT / 'phrases_model.h5'),
                                     monitor='val_accuracy', save_best_only=True),
    keras.callbacks.CSVLogger(str(LOG_OUT / 'phrases_training.csv')),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=2,
)

## 9. Training curves
Loss and accuracy per epoch — for the report.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.savefig(LOG_OUT / 'phrases_training_curves.png', dpi=150)
plt.show()

## 10. Evaluate on the held-out test set
Top-1, Top-3, Top-5 accuracy + per-class precision/recall/F1. For ISLR-style problems, top-3 and top-5 are useful: if the model's top guess is wrong but the correct sign is in the top 5, the demo can show 'did you mean … ?' suggestions.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred_proba = model.predict(X_test.astype(np.float32), batch_size=BATCH, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

test_acc = accuracy_score(y_test, y_pred)
top3 = np.mean([y_test[i] in np.argsort(y_pred_proba[i])[-3:] for i in range(len(y_test))])
top5 = np.mean([y_test[i] in np.argsort(y_pred_proba[i])[-5:] for i in range(len(y_test))])

print(f'Top-1 accuracy: {test_acc:.4f}')
print(f'Top-3 accuracy: {top3:.4f}')
print(f'Top-5 accuracy: {top5:.4f}')

label_names = [idx_to_label[i] for i in sorted(idx_to_label)]
report = classification_report(y_test, y_pred, target_names=label_names,
                                output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
report_df.to_csv(LOG_OUT / 'phrases_classification_report.csv')
print('\nPer-class report:')
print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

## 11. Confusion matrix
30×30 heatmap — same diagnostic value as Phase 4a's. Off-diagonal hot spots = signs that look similar in motion (e.g. 'mom' and 'dad' have very similar trajectories, just different hand shapes).

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=label_names, yticklabels=label_names,
            cmap='Blues', ax=ax)
ax.set_title(f'Confusion Matrix — Phrase LSTM (test acc {test_acc:.3f}, top3 {top3:.3f})')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(LOG_OUT / 'phrases_confusion_matrix.png', dpi=150)
plt.show()

## 12. Weak signs and confused pairs

In [ ]:
weak = [(lab, report[lab]['f1-score']) for lab in label_names if report[lab]['f1-score'] < 0.80]
weak.sort(key=lambda x: x[1])
print('Weak signs (F1 < 0.80):')
for lab, f1 in weak:
    print(f'  {lab:10s} F1={f1:.3f}')
if not weak:
    print('  (none)')

print('\nMost-confused pairs (>10% misclassified):')
for i, true_lab in enumerate(label_names):
    total = cm[i].sum()
    if total == 0: continue
    for j, pred_lab in enumerate(label_names):
        if i != j and cm[i][j] > 0.10 * total:
            pct = 100 * cm[i][j] / total
            print(f'  {true_lab} → {pred_lab}: {cm[i][j]} ({pct:.1f}%)')

## 13. Convert to TFLite
LSTMs need slightly different conversion — we enable `SELECT_TF_OPS` so TFLite can fall back to TensorFlow ops for any LSTM internals it doesn't have a native TFLite op for. The resulting model is larger than a Dense-only TFLite but still under a few MB.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter._experimental_lower_tensor_list_ops = False
tflite_bytes = converter.convert()

tflite_path = MODEL_OUT / 'phrases_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_bytes)
print(f'Saved {tflite_path}, {tflite_path.stat().st_size/1e6:.2f} MB')

## 14. Outputs and deliverables

In [ ]:
for p in sorted(OUT.rglob('*')):
    if p.is_file():
        print(f'{p.relative_to(OUT)}  —  {p.stat().st_size/1e3:.1f} KB')